In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import FeatureUnion
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# Load data
file_path = '/content/final3.xlsx'
df = pd.read_excel(file_path)

# Encode labels
le = LabelEncoder()
df["Specialist_Encoded"] = le.fit_transform(df["Specialist"])

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    df["Symptom"], df["Specialist_Encoded"], test_size=0.2, random_state=42, stratify=df["Specialist_Encoded"]
)

# Vectorizers
word_vectorizer = TfidfVectorizer(analyzer='word', ngram_range=(1,3), max_features=10000)
char_vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), max_features=10000)
combined = FeatureUnion([("word", word_vectorizer), ("char", char_vectorizer)])

X_train_vec = combined.fit_transform(X_train)
X_test_vec = combined.transform(X_test)

# Train Random Forest
rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42)
rf.fit(X_train_vec, y_train)

# Calibrate Random Forest probabilities
calibrated_rf = CalibratedClassifierCV(rf, method='isotonic', cv=5)
calibrated_rf.fit(X_train_vec, y_train)

# Predict and evaluate calibrated model
y_pred = calibrated_rf.predict(X_test_vec)
print("Accuracy (Calibrated RF):", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred, target_names=le.classes_))

# Calculate top-3 accuracy
y_proba = calibrated_rf.predict_proba(X_test_vec)
top_3_preds = np.argsort(y_proba, axis=1)[:, -3:]
top_3_correct = [y_test.iloc[i] in top_3_preds[i] for i in range(len(y_test))]
top_3_accuracy = np.mean(top_3_correct)
print(f"Top-3 Accuracy (Calibrated RF): {top_3_accuracy:.4f}")


Accuracy (Calibrated RF): 0.846276765739853
Classification Report:
                            precision    recall  f1-score   support

                Allergist       0.82      0.84      0.83        90
             Cardiologist       0.82      0.75      0.78       106
                  Dentist       0.90      0.95      0.92       165
            Dermatologist       0.94      0.95      0.94       219
            Diabetologist       0.88      0.83      0.86       101
           ENT Specialist       0.90      0.89      0.89       183
          Endocrinologist       0.76      0.62      0.68       131
       Gastroenterologist       0.76      0.77      0.77       110
        General Physician       0.70      0.55      0.61       152
             Gynecologist       0.80      0.85      0.83       101
             Hematologist       0.83      0.78      0.81       110
             Hepatologist       0.82      0.85      0.83        80
             Nephrologist       0.86      0.92      0.89    

In [ ]:
import joblib

# Save the trained calibrated RF model
joblib.dump(calibrated_rf, 'calibrated_rf_model.joblib')

# Save the combined TF-IDF vectorizer (word + char)
joblib.dump(combined, 'tfidf_vectorizer_combined.joblib')

# Save the label encoder for target decoding
joblib.dump(le, 'label_encoder.joblib')

print("Model and preprocessors saved successfully!")


Model and preprocessors saved successfully!


In [ ]:
import joblib

# Load saved objects
calibrated_rf = joblib.load('calibrated_rf_model.joblib')
combined = joblib.load('tfidf_vectorizer_combined.joblib')
le = joblib.load('label_encoder.joblib')

print("Model and preprocessors loaded successfully!")


In [ ]:
def predict_specialist(symptom_text):
    # Transform text using saved vectorizer
    X_vec = combined.transform([symptom_text])
    # Predict encoded class
    pred_encoded = calibrated_rf.predict(X_vec)
    # Decode class label
    pred_label = le.inverse_transform(pred_encoded)
    return pred_label[0]


In [ ]:
new_symptom = "fever"
predicted_specialist = predict_specialist(new_symptom)
print(f"Recommended Specialist: {predicted_specialist}")


Recommended Specialist: General Physician
